In [ ]:
import numpy as np
import pandas as pd
import os
from random import seed
from data_splitter import split_dataframe, concat_from_manifest
  
seed(1121)

In [ ]:
CHUNKS_DIR = "../Data"
MANIFEST   = f"{CHUNKS_DIR}/Dataset_manifest.txt"

In [11]:
data_path = '../Data/training_setA'
patient_path = os.path.join(data_path,"training")

In [3]:
patient_id = sorted(os.listdir(os.path.join(data_path,"training")))

In [ ]:
len_train = round(0.7*len(patient_id))+1
len_val = round(0.15*len(patient_id))
len_test = round(0.15*len(patient_id))
len_train + len_val + len_test == len(patient_id)

AA
AA
AA
AA
AA


In [5]:
len_train

14236

In [7]:
import random
train_id = random.sample(patient_id, len_train)
val_id = random.sample(sorted(set(patient_id) - set(train_id)), len_val)
test_id = set(patient_id) - set(train_id) - set(val_id)

## raw split

In [8]:
data_train = data_path + '/raw/training/'
data_val = data_path + '/raw/validation/'
data_test = data_path + '/raw/test/'

In [9]:
os.makedirs(data_train,exist_ok=True)
os.makedirs(data_val,exist_ok=True)
os.makedirs(data_test,exist_ok=True)

In [12]:
for p in train_id:
    df = pd.read_csv(patient_path + '/' + p, sep = "|")
    df.to_csv(data_train  + p, sep='|', index = False) 

In [13]:
for p in val_id:
    df = pd.read_csv(patient_path + '/' + p, sep = "|")
    df.to_csv(data_val + p, sep='|', index = False) 

In [14]:
for p in test_id:
    df = pd.read_csv(patient_path + '/' + p, sep = "|")
    df.to_csv(data_test + p, sep='|', index = False)

## Baseline

In [15]:
# function to fill missing values
def impute_missing_vals(df, attributes):

    """
    function that imputes missing values.
    
    @param df: dataframe that has missing values to be
               imputed
           attributes: list of String, attributes of dataframe
    @return df_clean: dataframe without missing values

    """
    
    """
    fill missing values by the closest values first
    ffill to fill missing values in the tail
    bfill to fill missing values in the head
    """
    # copy df
    df_clean = df.copy()
    for att in attributes:
        if df_clean[att].isnull().sum() == len(df_clean):
            df_clean[att] = df_clean[att].fillna(0)
        elif df_clean[att].isnull().sum() == len(df_clean) - 1:
            df_clean[att] = df_clean[att].ffill().bfill()
        else:
            df_clean[att] = df_clean[att].interpolate(method='nearest', limit_direction='both')
            df_clean[att] = df_clean[att].ffill().bfill()
    
    return df_clean

In [ ]:
# impute missing values and create clean dfs for all patients
for p in patient_id:
    
    # read in patient data
    df = pd.read_csv(patient_path + '/' + p, sep = "|")
    attributes = df.columns[:-1]
    
    # impute missing values
    df_clean = impute_missing_vals(df, attributes)
    
    # drop unit1 and unit2 with half missing values
    # because these two features have few information
    # drop EtCO2 with all missing values
    df_clean = df_clean.drop(['Unit1', 'Unit2', 'EtCO2'], axis=1)
    
    # save new patient data
    if p in train_id:
        save_path = data_path + '/baseline/train_baseline/'
        os.makedirs(save_path,exist_ok=True)
        df_clean.to_csv(save_path + p, sep='|', index = False)        
    
    elif p in val_id:
        save_path = data_path + '/baseline/val_baseline/'
        os.makedirs(save_path,exist_ok=True)
        df_clean.to_csv(save_path + p, sep='|', index = False)        
    
    else:
        
        save_path = data_path + '/baseline/test_baseline/'
        os.makedirs(save_path,exist_ok=True)
        df_clean.to_csv(save_path + p, sep='|', index = False)

In [ ]:
train = concat_from_manifest(MANIFEST)
#train = pd.read_csv("../Data/Dataset.csv")

In [24]:
train.head()

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [32]:
td_num = []
for i in range(len(train_id)):
    td_num.append(int(train_id[i][1:7]))

print(td_num)

[6337, 2882, 4243, 581, 15249, 8143, 14592, 3617, 6927, 8082, 12747, 17010, 130, 18440, 16408, 3293, 10364, 3014, 6889, 1942, 332, 761, 7413, 8485, 1733, 10982, 19821, 13397, 10419, 2041, 17041, 14365, 19301, 15019, 13727, 1574, 16646, 624, 14568, 14216, 2066, 12479, 15547, 17243, 17039, 16119, 16514, 19966, 3222, 15474, 10559, 18015, 17314, 10521, 6407, 16563, 365, 17487, 19118, 8087, 1783, 9103, 15348, 18864, 16211, 13927, 13639, 15868, 6988, 5009, 4705, 3409, 4641, 5769, 14945, 678, 173, 4268, 1549, 14605, 885, 3899, 11996, 15551, 7971, 11613, 14899, 9818, 6346, 11203, 5656, 9285, 16861, 11202, 7689, 4212, 19201, 15932, 9694, 15959, 12819, 11029, 8762, 10962, 17578, 4691, 12499, 7117, 10928, 5432, 6825, 17382, 16562, 19190, 19452, 3148, 18346, 11086, 10974, 13691, 18395, 17392, 18292, 17755, 19618, 3060, 10375, 16441, 7232, 17346, 7407, 17418, 14255, 16250, 19192, 19870, 4939, 3110, 11951, 20640, 7601, 12712, 11394, 5460, 860, 17665, 8492, 7592, 437, 4314, 3763, 10462, 11175, 16273,

In [33]:
train_concate = train[train.Patient_ID.isin(td_num)]

train_concate.to_csv("../Data/train_concate.csv",sep = '|')

In [ ]:
train_concate = pd.read_csv("../Data/train_concate.csv", sep = '|')

,Unnamed: 0.1,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,43,0,0,75.5,95.0,37.44,135.0,88.33,NaN,24.5,...,NaN,NaN,46.32,1,NaN,NaN,-0.04,4,0,16153
1,44,1,1,84.0,95.0,37.22,124.0,81.33,NaN,22.0,...,NaN,NaN,46.32,1,NaN,NaN,-0.04,5,0,16153
2,45,2,2,89.0,96.0,NaN,132.0,84.00,NaN,24.0,...,NaN,NaN,46.32,1,NaN,NaN,-0.04,6,0,16153
3,46,3,3,85.0,95.0,NaN,140.0,87.33,NaN,22.0,...,NaN,NaN,46.32,1,NaN,NaN,-0.04,7,0,16153
4,47,4,4,83.0,94.0,NaN,108.0,76.67,NaN,21.0,...,NaN,260.0,46.32,1,NaN,NaN,-0.04,8,0,16153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552834,790162,27,27,92.0,98.0,NaN,93.0,70.00,56.0,15.0,...,NaN,NaN,61.31,0,NaN,NaN,-8.17,33,0,11204
552835,790163,28,28,89.0,98.0,NaN,90.0,69.00,55.0,14.0,...,NaN,NaN,61.31,0,NaN,NaN,-8.17,34,0,11204
552836,790164,29,29,94.0,98.0,35.83,92.0,69.00,55.0,14.0,...,NaN,NaN,61.31,0,NaN,NaN,-8.17,35,0,11204
552837,790165,30,30,98.0,97.0,NaN,94.0,70.00,55.0,13.0,...,NaN,NaN,61.31,0,NaN,NaN,-8.17,36,0,11204


In [35]:
train_concate.shape

(552839, 44)

In [37]:
val_num = []
for i in range(len(val_id)):
    val_num.append(val_id[i][1:7])
    
val_concate = train[train.Patient_ID.isin(val_num)]

In [38]:
val_concate.to_csv('../Data/val_concate.csv', sep = '|')

In [ ]:
pd.read_csv('../Data/val_concate.csv', sep = '|')

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,75.91,0,0.0,1.0,-98.60,1,0,2
1,61.0,99.0,36.44,124.0,65.0,43.0,17.5,NaN,NaN,NaN,...,NaN,NaN,75.91,0,0.0,1.0,-98.60,2,0,2
2,64.0,98.0,NaN,125.0,64.0,41.0,27.0,NaN,NaN,NaN,...,NaN,NaN,75.91,0,0.0,1.0,-98.60,3,0,2
3,56.0,100.0,NaN,123.0,65.0,41.0,9.0,NaN,NaN,NaN,...,NaN,NaN,75.91,0,0.0,1.0,-98.60,4,0,2
4,66.0,99.0,NaN,120.0,67.0,43.0,23.0,NaN,NaN,NaN,...,NaN,NaN,75.91,0,0.0,1.0,-98.60,5,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118868,55.0,96.0,37.67,100.0,77.0,NaN,18.0,NaN,NaN,NaN,...,NaN,NaN,38.02,1,NaN,NaN,-0.03,11,0,20630
118869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,38.02,1,NaN,NaN,-0.03,12,0,20630
118870,60.0,97.0,NaN,100.0,83.0,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,38.02,1,NaN,NaN,-0.03,13,0,20630
118871,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,38.02,1,NaN,NaN,-0.03,14,0,20630


In [40]:
test_num = []
for i in range(len(test_id)):
    test_num.append(list(test_id)[i][1:7])
    
test_concate = train[train.Patient_ID.isin(test_num)]

In [41]:
test_concate.to_csv('../Data/test_concate.csv', sep = '|', index = False)

In [ ]:
pd.read_csv('../Data/test_concate.csv', sep = '|')

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,45.76,0,1.0,0.0,-0.03,1,0,17
1,79.0,98.0,NaN,134.0,81.33,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,45.76,0,1.0,0.0,-0.03,2,0,17
2,81.0,98.0,NaN,121.0,80.33,NaN,16.0,NaN,NaN,NaN,...,NaN,NaN,45.76,0,1.0,0.0,-0.03,3,0,17
3,88.0,100.0,NaN,131.0,77.00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,45.76,0,1.0,0.0,-0.03,4,0,17
4,86.0,100.0,NaN,143.0,89.00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,45.76,0,1.0,0.0,-0.03,5,0,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118234,83.0,94.0,37.5,149.0,83.00,55.0,17.0,NaN,NaN,NaN,...,NaN,NaN,74.53,0,0.0,1.0,-59.09,23,0,20640
118235,74.0,93.0,37.5,147.0,77.00,50.0,15.0,NaN,NaN,NaN,...,NaN,NaN,74.53,0,0.0,1.0,-59.09,24,0,20640
118236,74.0,95.0,37.5,138.0,74.00,52.0,15.0,NaN,NaN,NaN,...,NaN,NaN,74.53,0,0.0,1.0,-59.09,25,0,20640
118237,71.0,97.0,37.4,135.0,73.33,59.0,14.0,NaN,NaN,NaN,...,NaN,NaN,74.53,0,0.0,1.0,-59.09,26,0,20640
